# 07 - Model Comparison

In this notebook, we will load the models that we've trained and compare them to decide which one will be our final model.

To evaluate our models, we will compute F1-score on the same held-out test set.



In [2]:
import sys
sys.path.append("..")

import joblib
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from src.data_loading import load_titanic_data
from src.data_preprocessing import fill_missing_values, encode_categorical, FEATURES, TARGET

## Rebuild the same test set

Same loading, preprocessing, and split parameters that we used when we were training our models, so this reproduces the exact same `X_test`/`y_test`.

In [3]:
df = load_titanic_data()
df = fill_missing_values(df)
df = encode_categorical(df)

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Load the trained models

In [4]:
models = {
    "Logistic Regression": joblib.load("../models/baseline_logistic_regression.pkl"),
    "Random Forest": joblib.load("../models/random_forest.pkl"),
    "XGBoost": joblib.load("../models/xgboost.pkl"),
}

## Compare F1 scores

In [5]:
f1_scores = {
    name: f1_score(y_test, model.predict(X_test))
    for name, model in models.items()
}

comparison = pd.Series(f1_scores, name="F1 score").sort_values(ascending=False)
comparison

Random Forest          0.751880
XGBoost                0.729927
Logistic Regression    0.691729
Name: F1 score, dtype: float64

## Pick the best model

For simplicity, we'll use F1-score as the only factor to decide which model we choose. 

In [6]:
best_model_name = comparison.idxmax()
best_f1 = comparison.max()

print(f"Best model: {best_model_name} (F1={best_f1:.3f})")

Best model: Random Forest (F1=0.752)


## Export the final model

Let's also save the winning model under a fixed filename (`final_model.pkl`).

In [7]:
best_model = models[best_model_name]
joblib.dump(best_model, "../models/final_model.pkl")

['../models/final_model.pkl']